# Ensemble of ConvNeXt, EfficientNetV2-S, BiT and HRNet

This notebook averages four completed runs without retraining: Exp1 (ConvNeXt-base, public score 0.97003), Exp3 (EfficientNetV2-S, 0.97015), Exp5 (BiT ResNetV2-50x1, 0.94035) and Exp7 (HRNet-W32). The Exp1 + Exp3 + Exp5 mean scored 0.98047, our best public result. HRNet is a fourth CNN family: it keeps several resolution branches alive in parallel and exchanges information between them, where the other three reduce the resolution step by step. On the test set it differs from each member in 272–314 cells, as much as the earlier members differ from each other (219–291), so it should add information. This is ensembling, not stacking.

By default the notebook only reads saved probabilities and scores them. Test inference runs only if a member has no saved test probabilities and `STAGE2_RUN_TEST_INFERENCE=1`. The submission CSV and the run record are written only with `BUILD_SUBMISSION=True`, and never over an existing file. Kaggle upload is always manual.

In [1]:
from __future__ import annotations

import json
import os
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import torch

RUN_TEST_INFERENCE = True
BUILD_SUBMISSION = True  # Also saves the ensemble's fold-0 probabilities and run record.
SUBMIT_TO_KAGGLE = False  # No submission API call exists in this notebook.

RUN_ID = 'ensemble_exp1_exp3_exp5_exp7_mean_fold0'
VALIDATION_FOLD = 0
THRESHOLD = 0.25
TEST_BATCH_SIZE = 8
# Experiment folder -> completed run ID. Add a member here once its fold-0 run exists.
MEMBERS = {
    'exp1': 'convnext_standard_fold0_seed42_20261018_full1',
    'exp3': 'efficientnetv2s_fold0_seed42',
    'exp5': 'bit_r50x1_fold0_seed42',
    'exp07': 'hrnet_w32_finetune_fold0_seed42',
}
# The best ensemble so far (public 0.98047) and its members, for the comparison below.
PREVIOUS_BEST_MEMBERS = ('exp1', 'exp3', 'exp5')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', DEVICE, '| test inference:', RUN_TEST_INFERENCE, '| build submission:', BUILD_SUBMISSION)

Device: mps | test inference: True | build submission: True


## Load the members' fold-0 predictions

Each completed run saved its probabilities for the 2,452 fold-0 images. We join them by image ID, re-score them, and require the result to match the score in the run record. The members must share the fold file, label order and threshold. Otherwise the comparison would not be on equal terms. All four were trained on folds 1–4, each from its own head: Exp1 from Exp0's head, Exp3 and Exp5 from their own warm-ups, Exp7 from the local linear head on frozen HRNet features.

In [2]:
start = Path(os.environ.get('STAGE2_ROOT', Path.cwd())).resolve()
ROOT = next((path for path in (start, *start.parents) if (path / 'folds.csv').is_file() and (path / 'stage2' / 'experiment.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Launch Jupyter from this repository or a subdirectory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from stage2.ensemble import load_finetuned_network, mean_probabilities, read_probabilities, write_test_probabilities
from stage2.experiment import LABEL_COLUMNS, ImageDataset, form_submission, read_csv, score_probabilities, split_rows, write_probabilities, write_run_record, write_submission
from stage2.run_records import file_sha256
from stage2.train import predict_probabilities

DATA_DIR = ROOT / 'data'
_, validation_rows = split_rows(DATA_DIR, ROOT / 'folds.csv', VALIDATION_FOLD)
validation_ids = [row['image_id'] for row in validation_rows]
validation_targets = np.asarray([[int(row[label]) for label in LABEL_COLUMNS] for row in validation_rows])
fold_sha256 = file_sha256(ROOT / 'folds.csv')

records, checkpoints, validation_probabilities = {}, {}, {}
for name, run_id in MEMBERS.items():
    run_dir = ROOT / 'experiments' / name / 'results' / run_id
    record = json.loads((run_dir / f'{run_id}.json').read_text())
    if record['status'] != 'completed' or record['fold'] != VALIDATION_FOLD:
        raise ValueError(f'{run_id} is not a completed fold-{VALIDATION_FOLD} run.')
    if record['data']['fold_sha256'] != fold_sha256 or record['data']['label_columns'] != list(LABEL_COLUMNS):
        raise ValueError(f'{run_id} used a different fold file or label order.')
    if record['metrics']['threshold'] != THRESHOLD:
        raise ValueError(f'{run_id} was scored at another threshold.')
    probabilities = read_probabilities(run_dir / f'{run_id}_val.csv', validation_ids)
    rescored = score_probabilities(validation_targets, probabilities, THRESHOLD)['macro_f1']
    if not np.isclose(rescored, record['metrics']['macro_f1'], rtol=0, atol=1e-9):
        raise ValueError(f'{run_id}: re-scored macro F1 {rescored} differs from its record.')
    records[name] = record
    # The run record names the selected checkpoint; older runs keep it under experiments/<exp>/checkpoints/.
    checkpoints[name] = ROOT / record['artifacts'].get('best_checkpoint_local', '') if name == 'exp07' else ROOT / 'experiments' / name / 'checkpoints' / f'{run_id}_best.ckpt'
    validation_probabilities[name] = probabilities
    print(f'{name}: {run_id} | fold-0 macro F1 {rescored:.5f} (matches record)')

exp1: convnext_standard_fold0_seed42_20261018_full1 | fold-0 macro F1 0.97518 (matches record)
exp3: efficientnetv2s_fold0_seed42 | fold-0 macro F1 0.99831 (matches record)
exp5: bit_r50x1_fold0_seed42 | fold-0 macro F1 0.99851 (matches record)
exp07: hrnet_w32_finetune_fold0_seed42 | fold-0 macro F1 0.99733 (matches record)


## Compare single models and their mean on fold 0

We average probabilities with equal weights and keep the 0.25 threshold. We do not tune weights or thresholds: fold 0 has only 8–12 photos of each named pet, so a tuned weight would fit noise. On fold 0 one named-pet photo moves macro F1 by about 0.012, so differences smaller than that are not real differences. Exp3's, Exp5's and Exp7's checkpoints were selected on fold 0, so their fold-0 scores are optimistic. Exp5 scored 0.99851 here and 0.94035 on the leaderboard, so the fold-0 table is not a ranking of the models. It only shows whether the mean breaks anything.

The disagreement count shows how much the members differ. An ensemble cannot fix an image that every member gets wrong.

In [3]:
candidates = {**validation_probabilities, 'mean': mean_probabilities(list(validation_probabilities.values()))}
scores = {name: score_probabilities(validation_targets, probabilities, THRESHOLD) for name, probabilities in candidates.items()}
scores_at_half = {name: score_probabilities(validation_targets, probabilities, 0.5) for name, probabilities in candidates.items()}

header = f"{'model':<6} {'F1@0.25':>8} " + ' '.join(f'{label:>8}' for label in LABEL_COLUMNS) + f" {'AP':>8} {'F1@0.5':>8}"
print(header)
print('-' * len(header))
for name, metrics in scores.items():
    per_label = ' '.join(f"{metrics['per_label_f1'][label]:>8.4f}" for label in LABEL_COLUMNS)
    print(f"{name:<6} {metrics['macro_f1']:>8.5f} {per_label} {metrics['macro_ap']:>8.5f} {scores_at_half[name]['macro_f1']:>8.5f}")

predictions = {name: probabilities >= THRESHOLD for name, probabilities in candidates.items()}
member_names = list(MEMBERS)
disagreement = np.any([predictions[name] != predictions[member_names[0]] for name in member_names[1:]], axis=0)
print('\nImage-label cells where the members disagree:', int(disagreement.sum()),
      {label: int(disagreement[:, index].sum()) for index, label in enumerate(LABEL_COLUMNS)})
for name in candidates:
    false_positives = (predictions[name] & ~validation_targets.astype(bool)).sum(axis=0)
    false_negatives = (~predictions[name] & validation_targets.astype(bool)).sum(axis=0)
    print(f'{name:<6} errors per label (FP/FN):', {label: f'{fp}/{fn}' for label, fp, fn in zip(LABEL_COLUMNS, false_positives, false_negatives)})

model   F1@0.25      Cat      Dog    Moris    Motya  Biatrix       AP   F1@0.5
------------------------------------------------------------------------------
exp1    0.97518   0.9966   0.9970   1.0000   0.9412   0.9412  0.99443  0.97509
exp3    0.99831   0.9971   0.9945   1.0000   1.0000   1.0000  0.99998  0.99832
exp5    0.99851   0.9970   0.9955   1.0000   1.0000   1.0000  0.99971  0.98779
exp07   0.99733   0.9941   0.9925   1.0000   1.0000   1.0000  0.99995  0.99722
mean    0.99832   0.9956   0.9960   1.0000   1.0000   1.0000  0.99999  0.99881

Image-label cells where the members disagree: 49 {'Cat': 17, 'Dog': 30, 'Moris': 0, 'Motya': 1, 'Biatrix': 1}
exp1   errors per label (FP/FN): {'Cat': '5/2', 'Dog': '4/2', 'Moris': '0/0', 'Motya': '0/1', 'Biatrix': '1/0'}
exp3   errors per label (FP/FN): {'Cat': '5/1', 'Dog': '6/5', 'Moris': '0/0', 'Motya': '0/0', 'Biatrix': '0/0'}
exp5   errors per label (FP/FN): {'Cat': '4/2', 'Dog': '5/4', 'Moris': '0/0', 'Motya': '0/0', 'Biatrix': '0/0'}


### Named-pet mistakes

Every fold-0 image where at least one model makes a named-pet mistake, with each model's probability. This shows whether the mean rescues a member's miss or adds a new one.

In [4]:
named_pets = [LABEL_COLUMNS.index(label) for label in ('Moris', 'Motya', 'Biatrix')]
print(f"{'image_id':<14} {'label':<8} {'true':>4} " + ' '.join(f'{name:>7}' for name in candidates))
for row_index, image_id in enumerate(validation_ids):
    for label_index in named_pets:
        truth = bool(validation_targets[row_index, label_index])
        if any(predictions[name][row_index, label_index] != truth for name in candidates):
            values = ' '.join(f'{candidates[name][row_index, label_index]:>7.3f}' for name in candidates)
            print(f'{image_id[:12]:<14} {LABEL_COLUMNS[label_index]:<8} {int(truth):>4} {values}')

image_id       label    true    exp1    exp3    exp5   exp07    mean
5be84f949fb2   Motya       1   0.024   1.000   1.000   1.000   0.756
520c9a44013a   Biatrix     0   0.642   0.174   0.054   0.052   0.231


## Reuse the saved test probabilities

Each member's test probabilities are saved next to its run as `experiments/<exp>/results/<run-id>_test/` (the earlier submissions kept only 0/1 labels, which cannot be averaged). All four members already have them, so this cell reads them and runs no inference. A cached file is reused only if its record carries the same checkpoint hash. If a file were missing, the member would be predicted again, which needs `STAGE2_RUN_TEST_INFERENCE=1` and the member's checkpoint.

As a reproducibility check, we threshold each member's probabilities at 0.25 and compare them with the CSV that was scored on Kaggle. A few differing cells are expected only from arithmetic differences near the threshold.

In [5]:
test_rows = read_csv(DATA_DIR / 'test.csv', ('image_id',))
test_ids = [row['image_id'] for row in test_rows]
git_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip() if (ROOT / '.git').exists() else None
source_files = (ROOT / 'notebooks' / '18_ensemble_exp1_exp3_exp5_exp7.ipynb', ROOT / 'stage2' / 'ensemble.py', ROOT / 'stage2' / 'experiment.py')
source_sha256 = {str(path.relative_to(ROOT)): file_sha256(path) for path in source_files if path.is_file()}

test_probabilities = {}
for name, run_id in MEMBERS.items():
    record = records[name]
    cache_dir = ROOT / 'experiments' / name / 'results' / f'{run_id}_test'
    cache_csv, cache_json = cache_dir / f'{run_id}_test.csv', cache_dir / f'{run_id}_test.json'
    if cache_csv.is_file():
        cached = json.loads(cache_json.read_text())
        if cached['source_checkpoint_sha256'] != record['artifacts']['best_checkpoint_sha256']:
            raise ValueError(f'{cache_csv} was made from another checkpoint.')
        test_probabilities[name] = read_probabilities(cache_csv, test_ids)
        print(f'{name}: reused saved test probabilities.')
        continue
    if not RUN_TEST_INFERENCE:
        print(f'{name}: no saved test probabilities; set STAGE2_RUN_TEST_INFERENCE=1 to predict them.')
        continue
    from torch.utils.data import DataLoader

    network = load_finetuned_network(record, checkpoints[name])
    test_data = ImageDataset(test_rows, DATA_DIR / 'test', record['data']['image_size'], record['data']['mean'], record['data']['std'])
    test_loader = DataLoader(test_data, batch_size=TEST_BATCH_SIZE, shuffle=False, num_workers=0)
    print(f'{name}: predicting {len(test_rows)} test images on {DEVICE}...', flush=True)
    started = time.perf_counter()
    with torch.inference_mode():
        predicted_ids, probabilities = predict_probabilities(network, test_loader, DEVICE)
    seconds = time.perf_counter() - started
    if predicted_ids != test_ids:
        raise ValueError('Test loader changed image order.')
    write_test_probabilities(cache_csv, predicted_ids, probabilities)
    write_run_record(cache_json, {
        'run_id': f'{run_id}_test', 'experiment': record['experiment'], 'status': 'test_predicted',
        'source_run_id': run_id, 'source_checkpoint_sha256': record['artifacts']['best_checkpoint_sha256'],
        'test_images': len(predicted_ids), 'transform': record['data']['transform'] + '; no test-time augmentation',
        'git_commit': git_commit, 'source_sha256': source_sha256, 'device': str(DEVICE), 'prediction_seconds': seconds,
        'versions': {'python': platform.python_version(), 'torch': torch.__version__},
    })
    test_probabilities[name] = probabilities
    print(f'{name}: saved {cache_csv.relative_to(ROOT)} in {seconds / 60:.1f} min.')
    del network
    if DEVICE.type == 'mps':
        torch.mps.empty_cache()
    elif DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

scored_submissions = {'exp1': 'exp1_convnext_standard_fold0_025.csv', 'exp3': 'exp3_efficientnetv2s_fold0_seed42_025.csv',
                      'exp5': 'exp5_bit_r50x1_fold0_seed42_025.csv', 'exp07': 'exp07_hrnet_w32_fold0_seed42_025.csv'}
for name, probabilities in test_probabilities.items():
    path = ROOT / 'submissions' / scored_submissions.get(name, '')
    if path.is_file():
        scored = read_probabilities(path, test_ids)
        changed = (probabilities >= THRESHOLD) != scored.astype(bool)
        print(f'{name}: {int(changed.sum())} of {changed.size} cells differ from {path.name}',
              {label: int(changed[:, index].sum()) for index, label in enumerate(LABEL_COLUMNS)})

exp1: reused saved test probabilities.
exp3: reused saved test probabilities.
exp5: reused saved test probabilities.
exp07: reused saved test probabilities.
exp1: 0 of 93365 cells differ from exp1_convnext_standard_fold0_025.csv {'Cat': 0, 'Dog': 0, 'Moris': 0, 'Motya': 0, 'Biatrix': 0}
exp3: 0 of 93365 cells differ from exp3_efficientnetv2s_fold0_seed42_025.csv {'Cat': 0, 'Dog': 0, 'Moris': 0, 'Motya': 0, 'Biatrix': 0}
exp5: 0 of 93365 cells differ from exp5_bit_r50x1_fold0_seed42_025.csv {'Cat': 0, 'Dog': 0, 'Moris': 0, 'Motya': 0, 'Biatrix': 0}
exp07: 0 of 93365 cells differ from exp07_hrnet_w32_fold0_seed42_025.csv {'Cat': 0, 'Dog': 0, 'Moris': 0, 'Motya': 0, 'Biatrix': 0}


## Which ensemble? Compare the candidates on the test set

Fold 0 cannot rank these combinations: every one of them makes at most one or two named-pet mistakes there. The test set is where they differ, so we count how many cells each candidate changes compared with the current best (the Exp1 + Exp3 + Exp5 mean, public 0.98047). A named-pet cell is worth about 0.0007 macro F1 on the test set and a Cat or Dog cell about 0.00001, so only the named-pet changes can move the score noticeably.

The second table follows every named-pet change that adding HRNet makes, with the members that flagged the cell. In notebook 13 this showed why a weak member can still help: every cell it added was flagged by that member and every cell it removed was not. Test labels are hidden, so this shows who voted for a change, not whether the change is right.

In [6]:
if len(test_probabilities) != len(MEMBERS):
    raise ValueError('Test probabilities are missing for some members; run notebook cell 10 with inference enabled.')
member_labels = {name: probabilities >= THRESHOLD for name, probabilities in test_probabilities.items()}
previous = list(PREVIOUS_BEST_MEMBERS)
candidate_sets = {
    'mean 1+3+5 (public 0.98047)': previous,
    'mean 1+3+5+7 (this notebook)': list(MEMBERS),
    'mean 1+3+7': ['exp1', 'exp3', 'exp07'],
    'mean 1+3': ['exp1', 'exp3'],
}
candidate_test = {title: mean_probabilities([test_probabilities[name] for name in names]) for title, names in candidate_sets.items()}
reference = candidate_test['mean 1+3+5 (public 0.98047)'] >= THRESHOLD

header = f"{'candidate':<30} " + ' '.join(f'{label:>8}' for label in LABEL_COLUMNS) + f" {'vs best':>8} {'named':>6}"
print('Positives per label at', THRESHOLD, '| "vs best" = cells that differ from the 0.98047 mean, "named" = of those, named-pet cells')
print(header)
print('-' * len(header))
for title, probabilities in candidate_test.items():
    decided = probabilities >= THRESHOLD
    changed = decided != reference
    print(f'{title:<30} ' + ' '.join(f'{int(decided[:, index].sum()):>8}' for index in range(len(LABEL_COLUMNS)))
          + f' {int(changed.sum()):>8} {int(changed[:, named_pets].sum()):>6}')

new = candidate_test['mean 1+3+5+7 (this notebook)'] >= THRESHOLD
print('\nNamed-pet cells changed by adding HRNet to the 0.98047 mean, and which members flag them:')
for label_index in named_pets:
    label = LABEL_COLUMNS[label_index]
    for verb, cells in (('added', new[:, label_index] & ~reference[:, label_index]), ('removed', ~new[:, label_index] & reference[:, label_index])):
        flagged = {name: int((member_labels[name][:, label_index] & cells).sum()) for name in MEMBERS}
        print(f'  {label:<8} {verb:<8} {int(cells.sum()):>3}  flagged by', flagged)
votes = np.sum([member_labels[name][:, named_pets] for name in MEMBERS], axis=0)
for position, label_index in enumerate(named_pets):
    positive = new[:, label_index]
    print(f'  {LABEL_COLUMNS[label_index]:<8} positives {int(positive.sum()):>3}: ' + ', '.join(f'{count} members {int((positive & (votes[:, position] == count)).sum())}' for count in range(1, len(MEMBERS) + 1)))

Positives per label at 0.25 | "vs best" = cells that differ from the 0.98047 mean, "named" = of those, named-pet cells
candidate                           Cat      Dog    Moris    Motya  Biatrix  vs best  named
-------------------------------------------------------------------------------------------
mean 1+3+5 (public 0.98047)        7833     7718      167      137      185        0      0
mean 1+3+5+7 (this notebook)       7825     7700      165      138      184       58      8
mean 1+3+7                         7837     7709      169      135      186       92     19
mean 1+3                           7815     7699      171      131      187       81     24

Named-pet cells changed by adding HRNet to the 0.98047 mean, and which members flag them:
  Moris    added      1  flagged by {'exp1': 1, 'exp3': 0, 'exp5': 0, 'exp07': 1}
  Moris    removed    3  flagged by {'exp1': 3, 'exp3': 0, 'exp5': 0, 'exp07': 0}
  Motya    added      2  flagged by {'exp1': 1, 'exp3': 0, 'exp5': 1, 'exp

## Form the ensemble submission

The ensemble's test probabilities are the equal-weight mean of the four members', thresholded at 0.25. The table shows how many positives each label gets. A large jump in named-pet counts would need a closer look before submission.

With `BUILD_SUBMISSION=True`, the notebook writes the CSV to `submissions/`, plus the ensemble's fold-0 probabilities and a run record under `experiments/exp4/results/<run-id>/`, next to the other ensembles. Nothing is overwritten. Record the public score in the README after a manual upload.

In [11]:
if len(test_probabilities) == len(MEMBERS):
    ensemble_test = mean_probabilities([test_probabilities[name] for name in MEMBERS])
    print(f"{'labels':<8} " + ' '.join(f'{name:>7}' for name in (*MEMBERS, 'mean')))
    for index, label in enumerate(LABEL_COLUMNS):
        counts = [int((test_probabilities[name][:, index] >= THRESHOLD).sum()) for name in MEMBERS]
        counts.append(int((ensemble_test[:, index] >= THRESHOLD).sum()))
        print(f'{label:<8} ' + ' '.join(f'{count:>7}' for count in counts))
    ensemble_labels = ensemble_test >= THRESHOLD
    for name in MEMBERS:
        changed = ensemble_labels != (test_probabilities[name] >= THRESHOLD)
        print(f'Mean differs from {name} in {int(changed.sum())} cells:', {label: int(changed[:, index].sum()) for index, label in enumerate(LABEL_COLUMNS)})
else:
    ensemble_test = None
    print('Test probabilities are missing for some members; no ensemble test predictions yet.')

if BUILD_SUBMISSION:
    if ensemble_test is None:
        raise ValueError('Predict every member on the test set before building a submission.')
    result_dir = ROOT / 'experiments' / 'exp4' / 'results' / RUN_ID
    submission_path = ROOT / 'submissions' / f"exp4_{RUN_ID.removesuffix('_fold0')}_025.csv"
    for path in (submission_path, result_dir / f'{RUN_ID}.json', result_dir / f'{RUN_ID}_val.csv'):
        if path.exists():
            raise FileExistsError(f'Refusing to overwrite {path}')
    write_submission(submission_path, form_submission(DATA_DIR / 'sample_submission.csv', test_ids, ensemble_test, THRESHOLD))
    write_probabilities(result_dir / f'{RUN_ID}_val.csv', validation_ids, candidates['mean'], VALIDATION_FOLD)
    write_run_record(result_dir / f'{RUN_ID}.json', {
        'run_id': RUN_ID, 'experiment': '04_ensemble', 'status': 'completed', 'fold': VALIDATION_FOLD,
        'method': 'equal-weight mean of member probabilities, then fixed threshold',
        'members': [{'experiment': records[name]['experiment'], 'run_id': run_id,
                     'checkpoint_sha256': records[name]['artifacts']['best_checkpoint_sha256'],
                     'fold0_macro_f1': scores[name]['macro_f1']} for name, run_id in MEMBERS.items()],
        'data': {'fold_file': 'folds.csv', 'fold_sha256': fold_sha256, 'label_columns': list(LABEL_COLUMNS)},
        'metrics': {**scores['mean'], 'macro_f1_at_0_5': scores_at_half['mean']['macro_f1']},
        'artifacts': {'validation_probabilities_csv': str((result_dir / f'{RUN_ID}_val.csv').relative_to(ROOT)),
                      'submission_csv': str(submission_path.relative_to(ROOT))},
        'git_commit': git_commit, 'source_sha256': source_sha256,
        'public_leaderboard_score': None, 'error': None,
    })
    print('CSV ready for manual review:', submission_path.relative_to(ROOT))
    print('Run record:', (result_dir / f'{RUN_ID}.json').relative_to(ROOT))
else:
    print('No submission CSV built or uploaded.')

labels      exp1    exp3    exp5   exp07    mean
Cat         7781    7809    7794    7828    7825
Dog         7675    7659    7658    7655    7700
Moris        162     161     147     158     165
Motya        129     129     139     135     138
Biatrix      187     181     165     179     184
Mean differs from exp1 in 160 cells: {'Cat': 62, 'Dog': 51, 'Moris': 25, 'Motya': 15, 'Biatrix': 7}
Mean differs from exp3 in 143 cells: {'Cat': 48, 'Dog': 59, 'Moris': 8, 'Motya': 15, 'Biatrix': 13}
Mean differs from exp5 in 203 cells: {'Cat': 59, 'Dog': 84, 'Moris': 20, 'Motya': 19, 'Biatrix': 21}
Mean differs from exp07 in 199 cells: {'Cat': 73, 'Dog': 87, 'Moris': 17, 'Motya': 13, 'Biatrix': 9}


FileExistsError: Refusing to overwrite /Users/allakovalcuk/AI301/deep-learning-assignment2/experiments/exp4/results/ensemble_exp1_exp3_exp5_exp7_mean_fold0/ensemble_exp1_exp3_exp5_exp7_mean_fold0.json

## Results and conclusions

| Model | Fold-0 macro F1 at 0.25 | Public leaderboard |
|---|---:|---:|
| Exp1: ConvNeXt-base | 0.97518 | 0.97003 |
| Exp3: EfficientNetV2-S | 0.99831 | 0.97015 |
| Exp5: BiT ResNetV2-50x1 | 0.99851 | 0.94035 |
| Exp7: HRNet-W32 | 0.99733 | 0.96431 |
| **Mean of Exp1, Exp3 and Exp5** (notebook 13) | 0.98656 | **0.98047** |
| Mean of Exp1, Exp3, Exp5 and Exp7 (this notebook) | 0.99832 | 0.97816 |

**1. Adding HRNet made the leaderboard score worse: 0.97816, which is 0.0023 below the three-model mean.** That is about three named-pet cells' worth (roughly 0.0007 each). It is still above either Exp1 or Exp3 alone (+0.008), so averaging helps, but the fourth member took part of the gain away. The three-model mean (0.98047) stays our best submission. On fold 0 the four-model mean looked better (0.99832 against 0.98656, one Biatrix photo), so for this experiment fold 0 again pointed the wrong way.

**2. The change comes from eight named-pet cells, and most of them are cells that only one member called.** These are all the named-pet cells where the two means differ:

| Cell | Exp1 | Exp3 | Exp5 | Exp7 | 3-mean | 4-mean | Change |
|---|---:|---:|---:|---:|---:|---:|---|
| Moris | 0.800 | 0.057 | 0.006 | 0.083 | 0.288 | 0.236 | removed |
| Moris | 0.858 | 0.004 | 0.066 | 0.004 | 0.309 | 0.233 | removed |
| Moris | 0.955 | 0.003 | 0.005 | 0.001 | 0.321 | 0.241 | removed |
| Biatrix | 0.914 | 0.019 | 0.000 | 0.000 | 0.311 | 0.233 | removed |
| Motya | 0.277 | 0.028 | 0.476 | 0.001 | 0.260 | 0.195 | removed |
| Moris | 0.574 | 0.071 | 0.006 | 0.488 | 0.217 | 0.285 | added |
| Motya | 0.310 | 0.057 | 0.184 | 0.822 | 0.184 | 0.343 | added |
| Motya | 0.011 | 0.005 | 0.264 | 0.723 | 0.094 | 0.251 | added |

Four of the five removed cells were called by Exp1 alone (0.80 to 0.96, every other member 0.08 or less). A single member can add at most 1/4 = 0.25 to a four-model mean, so such a cell ends just under the 0.25 threshold (0.23 to 0.24). In the three-model mean the same member adds up to 1/3, and the cell passes (0.29 to 0.32). The three cells it added each need HRNet's vote. Because the score fell by about three cells' worth, most of the five removed cells were probably real pets, but the test labels are hidden, so this is our reading, not a measurement.

**3. The fixed threshold means a different voting rule for every ensemble size.** 0.25 was chosen for the first baseline (section 3 of the README), not for averaged probabilities. With three members, one confident vote (probability 0.75 or more) is enough. With four, it is not. The same effect lowered the stacking submission (0.96685) and may explain part of the full-data retraining result (0.97386). Notebook 13 already noted that the mean's decisions at 0.25 and at 0.5 differ in more cells than adding a member does.

**4. HRNet is a weaker member than BiT was a helper.** BiT scored lower alone (0.94035) but its votes settled cells where Exp1 and Exp3 disagreed, and the score rose. HRNet's own score (0.96431) is higher, but its three added cells did not make up for the five removed ones.

**5. Not tested.** We did not tune member weights or the threshold. Fold 0 has 8–10 photos of each named pet, three of the four members were selected on it, and tuning on the public leaderboard would fit its public part. A weighted mean or a lower threshold would change the same few named-pet cells, so it can only be judged by another submission.